In [1]:

import pandas as pd
import numpy as np

# Load the diabetes dataset
df = pd.read_csv("diabetes.csv")

# Display dataset information
print("Dataset loaded successfully!")
print("Shape of dataset:", df.shape)

print("\nFirst 5 rows:")
display(df.head())

print("\nColumn names:")
print(df.columns.tolist())

print("\nDataset information:")
df.info()


Dataset loaded successfully!
Shape of dataset: (768, 9)

First 5 rows:


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1



Column names:
['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome']

Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   768 non-null    int64  
 2   BloodPressure             768 non-null    int64  
 3   SkinThickness             768 non-null    int64  
 4   Insulin                   768 non-null    int64  
 5   BMI                       768 non-null    float64
 6   DiabetesPedigreeFunction  768 non-null    float64
 7   Age                       768 non-null    int64  
 8   Outcome                   768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB


In [2]:

# Check the number of zero values in each column

zero_columns = ['Glucose', 'BloodPressure',
                'SkinThickness', 'Insulin', 'BMI']

for col in zero_columns:
    print(col, ":", (df[col] == 0).sum())


Glucose : 5
BloodPressure : 35
SkinThickness : 227
Insulin : 374
BMI : 11


In [3]:

from sklearn.preprocessing import StandardScaler

# 1. Replace biologically invalid zeros with NaN
zero_columns = ['Glucose', 'BloodPressure',
                'SkinThickness', 'Insulin', 'BMI']

df[zero_columns] = df[zero_columns].replace(0, np.nan)

# 2. Median imputation
for col in zero_columns:
    df[col] = df[col].fillna(df[col].median())

print("Missing values after imputation:")
print(df.isnull().sum())

# 3. Detect and remove outliers using IQR
outlier_columns = df.drop(columns=['Outcome']).columns

Q1 = df[outlier_columns].quantile(0.25)
Q3 = df[outlier_columns].quantile(0.75)
IQR = Q3 - Q1

outlier_condition = (
    (df[outlier_columns] < (Q1 - 1.5 * IQR)) |
    (df[outlier_columns] > (Q3 + 1.5 * IQR))
).any(axis=1)

print("\nOutliers detected:", outlier_condition.sum())

df = df[~outlier_condition].copy()

# 4. Separate features and target
X = df.drop(columns=['Outcome'])
y = df['Outcome']

# 5. Standardize numerical features
scaler = StandardScaler()
X_scaled = pd.DataFrame(
    scaler.fit_transform(X),
    columns=X.columns,
    index=X.index
)

print("\nFinal dataset shape:", X_scaled.shape)
print("\nTarget distribution:")
print(y.value_counts())

print("\nPreprocessing completed successfully!")



Missing values after imputation:
Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64

Outliers detected: 393

Final dataset shape: (375, 8)

Target distribution:
Outcome
0    237
1    138
Name: count, dtype: int64

Preprocessing completed successfully!


In [4]:

from sklearn.feature_selection import RFE
from sklearn.ensemble import RandomForestClassifier

# 1. Feature Engineering
X_scaled['Glucose_BMI_Index'] = (
    X_scaled['Glucose'] * X_scaled['BMI']
)

print("New feature created: Glucose_BMI_Index")

# 2. Feature Selection using RFE
rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rfe = RFE(
    estimator=rf,
    n_features_to_select=5
)

X_selected = rfe.fit_transform(X_scaled, y)

# 3. Display selected features
selected_features = X_scaled.columns[rfe.support_]

print("\nSelected Features:")
print(list(selected_features))

print("\nFeature Rankings:")
for feature, rank in zip(X_scaled.columns, rfe.ranking_):
    print(feature, ":", rank)

print("\nOriginal number of features:", X_scaled.shape[1])
print("Selected number of features:", X_selected.shape[1])

print("\nFeature engineering and selection completed!")


New feature created: Glucose_BMI_Index

Selected Features:
['Glucose', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Glucose_BMI_Index']

Feature Rankings:
Pregnancies : 3
Glucose : 1
BloodPressure : 2
SkinThickness : 4
Insulin : 5
BMI : 1
DiabetesPedigreeFunction : 1
Age : 1
Glucose_BMI_Index : 1

Original number of features: 9
Selected number of features: 5

Feature engineering and selection completed!


In [5]:

# ==========================================
# STEP 9: RANDOM FOREST MODEL TRAINING
# V1 - Baseline
# V2 - Hyperparameter Tuned
# V3 - RFE Selected Features
# ==========================================

import os
import joblib
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import RFE
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, classification_report,
    confusion_matrix
)

# 1. Prepare features and target
X = df.drop(columns=['Outcome']).copy()
y = df['Outcome'].copy()

# Feature engineering
X['Glucose_BMI_Index'] = X['Glucose'] * X['BMI']

print("Feature engineering completed!")
print("Total features:", X.shape[1])

# 2. Split dataset into training and testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\nTraining samples:", len(X_train))
print("Testing samples:", len(X_test))

# 3. Standardization
scaler = StandardScaler()

X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)

X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)

# 4. Feature selection using RFE
rfe_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rfe = RFE(
    estimator=rfe_model,
    n_features_to_select=5
)

rfe.fit(X_train_scaled, y_train)

selected_features = X_train_scaled.columns[rfe.support_].tolist()

X_train_selected = X_train_scaled[selected_features]
X_test_selected = X_test_scaled[selected_features]

print("\nSelected Features:")
print(selected_features)

# 5. Define model versions
models = {}

# V1: Baseline Random Forest
models['RF_V1_Baseline'] = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

# Train V1
models['RF_V1_Baseline'].fit(X_train_scaled, y_train)

# V2: Hyperparameter tuning using RandomizedSearchCV
parameter_grid = {
    'n_estimators': [100, 150, 200, 250],
    'max_depth': [None, 5, 10, 15, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2']
}

random_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_distributions=parameter_grid,
    n_iter=12,
    cv=5,
    scoring='roc_auc',
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train_scaled, y_train)

models['RF_V2_Tuned'] = random_search.best_estimator_

print("\nBest parameters for V2:")
print(random_search.best_params_)

# V3: Tuned Random Forest with RFE-selected features
models['RF_V3_RFE'] = RandomForestClassifier(
    **random_search.best_params_,
    random_state=42
)

models['RF_V3_RFE'].fit(X_train_selected, y_train)

# 6. Evaluate all model versions
results = []

for name, model in models.items():

    if name == 'RF_V3_RFE':
        X_eval = X_test_selected
    else:
        X_eval = X_test_scaled

    predictions = model.predict(X_eval)
    probabilities = model.predict_proba(X_eval)[:, 1]

    accuracy = accuracy_score(y_test, predictions)
    precision = precision_score(y_test, predictions, zero_division=0)
    recall = recall_score(y_test, predictions, zero_division=0)
    f1 = f1_score(y_test, predictions, zero_division=0)
    auc = roc_auc_score(y_test, probabilities)

    results.append({
        'Model': name,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1_Score': f1,
        'ROC_AUC': auc
    })

    print("\n", "=" * 45)
    print("MODEL:", name)
    print("=" * 45)

    print(classification_report(
        y_test, predictions, zero_division=0
    ))

    print("Confusion Matrix:")
    print(confusion_matrix(y_test, predictions))

# 7. Performance comparison
results_df = pd.DataFrame(results)

print("\n========== MODEL PERFORMANCE COMPARISON ==========")
display(results_df.round(4))

# 8. Identify best model based on F1 score
best_index = results_df['F1_Score'].idxmax()
best_model_name = results_df.loc[best_index, 'Model']

print("\nBest model based on F1 Score:", best_model_name)

# 9. Save models and experiment artifacts
os.makedirs("models", exist_ok=True)

joblib.dump(models, "models/all_model_versions.pkl")
joblib.dump(scaler, "models/scaler.pkl")
joblib.dump(rfe, "models/rfe_selector.pkl")
joblib.dump(selected_features, "models/selected_features.pkl")

best_model = models[best_model_name]
joblib.dump(best_model, "models/best_model.pkl")

results_df.to_csv("models/model_comparison.csv", index=False)

print("\nAll model versions saved successfully!")
print("Model training and comparison completed!")


Feature engineering completed!
Total features: 9

Training samples: 300
Testing samples: 75

Selected Features:
['Glucose', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Glucose_BMI_Index']

Best parameters for V2:
{'n_estimators': 150, 'min_samples_split': 10, 'min_samples_leaf': 2, 'max_features': 'log2', 'max_depth': None}

MODEL: RF_V1_Baseline
              precision    recall  f1-score   support

           0       0.82      0.85      0.83        47
           1       0.73      0.68      0.70        28

    accuracy                           0.79        75
   macro avg       0.77      0.76      0.77        75
weighted avg       0.78      0.79      0.78        75

Confusion Matrix:
[[40  7]
 [ 9 19]]

MODEL: RF_V2_Tuned
              precision    recall  f1-score   support

           0       0.78      0.85      0.82        47
           1       0.71      0.61      0.65        28

    accuracy                           0.76        75
   macro avg       0.75      0.73      0.74       

,Model,Accuracy,Precision,Recall,F1_Score,ROC_AUC
0,RF_V1_Baseline,0.7867,0.7308,0.6786,0.7037,0.8226
1,RF_V2_Tuned,0.7600,0.7083,0.6071,0.6538,0.8305
2,RF_V3_RFE,0.7333,0.6818,0.5357,0.6000,0.8191



Best model based on F1 Score: RF_V1_Baseline

All model versions saved successfully!
Model training and comparison completed!
